# Dasar Machine Learning

## Course 3 Bagian a: Regresi Logistik dan Metrik Kesalahan

Latihan ini memakai data *Human Activity Recognition Using Smartphones*. Setiap baris berisi 561 fitur hasil pengukuran sensor ponsel, sedangkan kolom `Activity` menjadi target yang harus diprediksi.


## Tujuan

- Memuat dan memeriksa struktur data.
- Menghitung korelasi antarfitur dan menyaring pasangan yang sangat berkorelasi.
- Membagi data latih dan uji secara stratifikasi.
- Membandingkan regresi logistik tanpa regularisasi, dengan L1, dan dengan L2.
- Menilai model memakai accuracy, precision, recall, F1, ROC-AUC, serta confusion matrix.


## Menyiapkan pustaka

Seluruh perhitungan memakai `pandas`, `numpy`, `seaborn`, dan `scikit-learn`.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


## Soal 1: Memuat data dan memeriksa tipe kolom

Dataset diambil langsung dari alamat publik, lalu diperiksa tipe datanya. Semua kolom prediktor bertipe float, hanya kolom aktivitas yang berupa teks.

Agar proses latihan berjalan ringkas, hanya sebagian baris yang dipakai. Hapus baris `sample` bila ingin memakai seluruh data.


In [ ]:
DATA_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-ML241EN-SkillsNetwork/labs/datasets/"
    "Human_Activity_Recognition_Using_Smartphones_Data.csv"
)
SAMPLE_SIZE = 2000

dataset = pd.read_csv(DATA_URL, sep=",").sample(
    n=SAMPLE_SIZE, random_state=42
).reset_index(drop=True)
dataset.shape


In [ ]:
dataset.dtypes.value_counts()


In [ ]:
dataset.dtypes.tail()


Nilai seluruh fitur sudah diskalakan pada rentang -1 sampai 1, sehingga tidak perlu normalisasi tambahan.


In [ ]:
dataset.iloc[:, :-1].min().value_counts()


In [ ]:
dataset.iloc[:, :-1].max().value_counts()


Sebaran kelas aktivitas cukup seimbang, seperti terlihat pada hitungan berikut.


In [ ]:
dataset["Activity"].value_counts()


Classifier scikit-learn membutuhkan target berupa angka. Karena itu label aktivitas diubah memakai `LabelEncoder`.


In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
dataset["Activity"] = label_encoder.fit_transform(dataset["Activity"])
dataset["Activity"].sample(5, random_state=42)


## Soal 2: Korelasi antarfitur

Matriks korelasi dihitung untuk semua prediktor. Agar tidak ada pasangan yang dihitung dua kali, hanya segitiga atas matriks yang diambil, lalu pasangan dengan korelasi absolut di atas 0.8 ditampilkan.


In [ ]:
feature_columns = dataset.columns[:-1]
correlation_matrix = dataset[feature_columns].corr()

upper_rows, upper_cols = np.triu_indices_from(correlation_matrix, k=1)
correlation_pairs = pd.DataFrame(
    {
        "feature1": correlation_matrix.columns[upper_cols],
        "feature2": correlation_matrix.index[upper_rows],
        "correlation": correlation_matrix.values[upper_rows, upper_cols],
    }
)
correlation_pairs["abs_correlation"] = correlation_pairs["correlation"].abs()
correlation_pairs.head()


In [ ]:
sns.set_context("talk")
sns.set_style("white")

axis = correlation_pairs.abs_correlation.hist(bins=50, figsize=(12, 8))
axis.set(xlabel="Absolute Correlation", ylabel="Frequency")
plt.show()


In [ ]:
correlation_pairs.sort_values("correlation", ascending=False).query(
    "abs_correlation > 0.8"
)


## Soal 3: Pembagian data latih dan uji

Pembagian memakai `StratifiedShuffleSplit` agar proporsi setiap kelas aktivitas tetap terjaga pada kedua bagian data.


In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_index, test_index = next(
    splitter.split(dataset[feature_columns], dataset["Activity"])
)

X_train = dataset.loc[train_index, feature_columns]
y_train = dataset.loc[train_index, "Activity"]
X_test = dataset.loc[test_index, feature_columns]
y_test = dataset.loc[test_index, "Activity"]


In [ ]:
y_train.value_counts(normalize=True)


In [ ]:
y_test.value_counts(normalize=True)


## Soal 4: Tiga varian regresi logistik

Model pertama dilatih tanpa regularisasi. Dua model berikutnya memakai validasi silang internal untuk memilih nilai penalti terbaik, masing-masing dengan penalti L1 dan L2.


In [ ]:
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.multiclass import OneVsRestClassifier

MAX_ITER = 300

plain_model = OneVsRestClassifier(
    LogisticRegression(max_iter=MAX_ITER, solver="liblinear")
).fit(X_train, y_train)
l1_model = OneVsRestClassifier(
    LogisticRegressionCV(
        Cs=5, cv=3, penalty="l1", max_iter=MAX_ITER, solver="liblinear"
    )
).fit(X_train, y_train)
l2_model = OneVsRestClassifier(
    LogisticRegressionCV(
        Cs=5, cv=3, penalty="l2", max_iter=MAX_ITER, solver="liblinear"
    )
).fit(X_train, y_train)


## Soal 5: Membandingkan besar koefisien

Koefisien ketiga model disusun berdampingan. Setiap model menghasilkan enam set koefisien karena klasifikasinya multikelas.


In [ ]:
model_registry = {"lr": plain_model, "l1": l1_model, "l2": l2_model}

coefficient_table = pd.concat(
    [
        pd.DataFrame(
            np.vstack([estimator.coef_ for estimator in model.estimators_]).T,
            columns=pd.MultiIndex.from_product([[name], range(6)]),
        )
        for name, model in model_registry.items()
    ],
    axis=1,
)
coefficient_table.sample(10, random_state=42)


In [ ]:
figure, axes = plt.subplots(nrows=3, ncols=2)
axes = axes.flatten()
figure.set_size_inches(10, 10)

for position, axis in enumerate(axes):
    subset = coefficient_table.xs(position, level=1, axis=1)
    subset.plot(marker="o", ls="", ms=2.0, ax=axis, legend=False)
    axis.set(title=f"Coefficient Set {position}")

axes[0].legend(loc=4)
plt.tight_layout()
plt.show()


## Soal 6: Prediksi kelas dan probabilitas

Untuk setiap model disimpan kelas hasil prediksi dan probabilitas kelas terpilih.


In [ ]:
prediction_table = pd.concat(
    {
        name: pd.Series(model.predict(X_test), name=name)
        for name, model in model_registry.items()
    },
    axis=1,
)
probability_table = pd.concat(
    {
        name: pd.Series(model.predict_proba(X_test).max(axis=1), name=name)
        for name, model in model_registry.items()
    },
    axis=1,
)
prediction_table.head()


In [ ]:
probability_table.head()


## Soal 7: Metrik kesalahan

Karena target memiliki lebih dari dua kelas, precision, recall, dan F1 dihitung dengan rata-rata berbobot. ROC-AUC dihitung setelah target dibinerkan per kelas.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.preprocessing import label_binarize

class_labels = list(range(6))
metric_rows = []
confusion_matrices = {}

for name, model in model_registry.items():
    predictions = prediction_table[name]
    precision, recall, fscore, _ = precision_recall_fscore_support(
        y_test, predictions, average="weighted"
    )
    accuracy = accuracy_score(y_test, predictions)
    auc = roc_auc_score(
        label_binarize(y_test, classes=class_labels),
        label_binarize(predictions, classes=class_labels),
        average="weighted",
    )
    confusion_matrices[name] = confusion_matrix(y_test, predictions)
    metric_rows.append(
        pd.Series(
            {
                "precision": precision,
                "recall": recall,
                "fscore": fscore,
                "accuracy": accuracy,
                "auc": auc,
            },
            name=name,
        )
    )

metric_table = pd.concat(metric_rows, axis=1)
metric_table


## Soal 8: Confusion matrix setiap model

Confusion matrix ditampilkan sebagai heatmap agar pola kesalahan antarkelas mudah dibaca.


In [ ]:
figure, axes = plt.subplots(nrows=2, ncols=2)
axes = axes.flatten()
figure.set_size_inches(12, 10)
axes[-1].axis("off")

for axis, name in zip(axes[:-1], model_registry):
    sns.heatmap(confusion_matrices[name], ax=axis, annot=True, fmt="d")
    axis.set(title=name)

plt.tight_layout()
plt.show()


## Ringkasan

Regresi logistik dengan penalti L2 memberi keseimbangan terbaik antara akurasi dan kestabilan koefisien. Penalti L1 menyusutkan sebagian koefisien menjadi nol, sedangkan model tanpa regularisasi cenderung memakai seluruh fitur.
